# Forensic-Ready Explainable Intrusion Detection with Temporal Graph Modeling and Counterfactual Cyber Defense

## Introduction

This notebook implements a **forensic-ready and explainable intrusion detection pipeline** that combines four complementary layers:  
1. **supervised intrusion detection**,  
2. **explainable artificial intelligence (XAI)**,  
3. **temporal graph-based forensic analysis**, and  
4. **counterfactual cyber defense**.

The purpose of the notebook is not only to classify malicious traffic, but also to answer higher-level forensic and operational questions such as:

- Which attacks are detected reliably and which remain challenging?
- Which features drive the model's decisions?
- Which network entities appear most suspicious over time?
- How can suspicious activity be interpreted from a graph and campaign perspective?
- What minimal changes would shift a prediction from malicious to benign?

A deliberate design choice of this notebook is to **avoid generating many final figures and tables during the early experimentation stage**.  
Instead, the notebook writes a compact set of intermediate results to **one main machine-readable file** and **one optional flat log file** stored on Google Drive. These files act as an internal decision layer between experimentation and article preparation.

### Main output philosophy

During development, the notebook stores evidence such as:

- dataset summary,
- class balance,
- model performance metrics,
- confusion matrix values,
- ROC-AUC and calibration indicators,
- top SHAP features,
- local LIME explanations,
- temporal graph statistics,
- ranked suspicious nodes,
- counterfactual recommendations,
- notes about whether the current run appears strong enough for article-grade reporting.

This design makes it easier to:

- compare notebook runs,
- decide whether results are improving,
- postpone article-oriented plots until the methodology is mature,
- keep a compact, reproducible record of all core indicators.

### Target workflow

The notebook proceeds through the following stages:

1. Mount Google Drive and define reproducible output folders.
2. Register experiment metadata and create helper functions.
3. Load and validate the intrusion dataset.
4. Clean data and prepare features.
5. Split the data with stratification.
6. Train a strong baseline classifier.
7. Evaluate classification performance.
8. Compute explainability outputs with SHAP and LIME.
9. Build a temporal interaction graph for forensic analysis.
10. Rank suspicious nodes and summarize campaign patterns.
11. Generate counterfactual defensive recommendations.
12. Save all critical evidence to compact intermediate result files.

### Main files generated on Google Drive

The notebook writes primarily to:

- **`experiment_results.json`**: structured, hierarchical record of all important outputs,
- **`experiment_log.csv`**: optional flat experiment log for quick cross-run comparison.

Once the results become sufficiently strong, a second notebook version can transform these intermediate outputs into article-ready figures and tables.

## Text Cell 1 — Environment Setup and Google Drive Output Strategy

This cell imports the core Python libraries used throughout the notebook and mounts Google Drive in Google Colab.  
It also creates a clear folder structure dedicated to the project. The structure is intentionally simple and centered on **intermediate experimental evidence** rather than publication graphics.

At this stage, the notebook creates:

- a **base project folder** on Google Drive,
- a folder for **intermediate outputs**,
- a folder for **optional article assets** to be used later,
- two main result files:
  - `experiment_results.json`
  - `experiment_log.csv`

This organization supports a gradual workflow:
first verify whether the methodology is strong enough, then generate article-ready visuals only when justified.

In [76]:
# ============================================
# Cell 1: Environment setup and Google Drive
# ============================================

from pathlib import Path
import json
import os
import warnings
from datetime import datetime

import numpy as np
import pandas as pd

warnings.filterwarnings("ignore")

# Google Drive mount (for Colab)
try:
    from google.colab import drive
    drive.mount('/content/drive')
    IN_COLAB = True
except Exception:
    IN_COLAB = False
    print("Google Colab not detected. Running in local mode.")

# Project paths
BASE_DIR = Path("/content/drive/MyDrive/Outputs/Forensic_Ready_XAI_IDS") if IN_COLAB else Path.cwd() / "Forensic_Ready_XAI_IDS"
INTERMEDIATE_DIR = BASE_DIR / "intermediate_outputs"
ARTICLE_DIR = BASE_DIR / "article_assets"
DATA_DIR = BASE_DIR / "data"

for folder in [BASE_DIR, INTERMEDIATE_DIR, ARTICLE_DIR, DATA_DIR]:
    folder.mkdir(parents=True, exist_ok=True)

RESULTS_JSON = INTERMEDIATE_DIR / "experiment_results.json"
RESULTS_CSV = INTERMEDIATE_DIR / "experiment_log.csv"

print(f"Base directory: {BASE_DIR}")
print(f"Intermediate JSON: {RESULTS_JSON}")
print(f"Experiment log CSV: {RESULTS_CSV}")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Base directory: /content/drive/MyDrive/Outputs/Forensic_Ready_XAI_IDS
Intermediate JSON: /content/drive/MyDrive/Outputs/Forensic_Ready_XAI_IDS/intermediate_outputs/experiment_results.json
Experiment log CSV: /content/drive/MyDrive/Outputs/Forensic_Ready_XAI_IDS/intermediate_outputs/experiment_log.csv


## Text Cell 2 — Experiment Registry and Helper Functions

This cell defines the notebook's **result management layer**.  
Instead of scattering outputs across many files, we create helper functions that continuously append and update a central experimental record.

The idea is simple:

- every important result is written to a structured dictionary,
- the dictionary is serialized to `experiment_results.json`,
- a compact summary row is appended to `experiment_log.csv`.

This design supports comparison across runs and makes it easy to answer:
“Did the current model improve or not?”

The helper functions will store:

- metadata,
- dataset diagnostics,
- model configuration,
- performance metrics,
- confusion matrix values,
- graph indicators,
- XAI summaries,
- counterfactual summaries,
- qualitative status notes.

In [77]:
# ============================================
# Cell 2: Result registry helpers
# ============================================

RUN_ID = datetime.now().strftime("%Y%m%d_%H%M%S")

experiment_results = {
    "run_id": RUN_ID,
    "title": "Forensic-Ready Explainable Intrusion Detection with Temporal Graph Modeling and Counterfactual Cyber Defense",
    "created_at": datetime.now().isoformat(),
    "status_notes": [],
    "dataset": {},
    "preprocessing": {},
    "split": {},
    "model": {},
    "evaluation": {},
    "xai": {},
    "forensics": {},
    "counterfactuals": {},
}

def to_serializable(obj):
    if isinstance(obj, (np.integer,)):
        return int(obj)
    if isinstance(obj, (np.floating,)):
        return float(obj)
    if isinstance(obj, (np.ndarray,)):
        return obj.tolist()
    if isinstance(obj, (pd.Series,)):
        return obj.to_dict()
    if isinstance(obj, (pd.DataFrame,)):
        return obj.to_dict(orient="records")
    if isinstance(obj, (Path,)):
        return str(obj)
    return obj

def deep_convert(value):
    if isinstance(value, dict):
        return {k: deep_convert(v) for k, v in value.items()}
    if isinstance(value, list):
        return [deep_convert(v) for v in value]
    return to_serializable(value)

def update_results(section, payload):
    if section not in experiment_results:
        experiment_results[section] = {}
    if isinstance(experiment_results[section], dict) and isinstance(payload, dict):
        experiment_results[section].update(deep_convert(payload))
    else:
        experiment_results[section] = deep_convert(payload)

def add_status_note(note):
    experiment_results["status_notes"].append(
        {
            "timestamp": datetime.now().isoformat(),
            "note": str(note)
        }
    )

def save_results_json():
    with open(RESULTS_JSON, "w", encoding="utf-8") as f:
        json.dump(deep_convert(experiment_results), f, indent=2, ensure_ascii=False)

def append_experiment_log(row_dict):
    row_df = pd.DataFrame([deep_convert(row_dict)])
    if RESULTS_CSV.exists():
        row_df.to_csv(RESULTS_CSV, mode="a", index=False, header=False)
    else:
        row_df.to_csv(RESULTS_CSV, mode="w", index=False, header=True)

add_status_note("Experiment registry initialized.")
save_results_json()
print("Experiment registry is ready.")

FIGURES_DIR = ARTICLE_DIR / "figures"
TABLES_DIR = ARTICLE_DIR / "tables"
REPORTS_DIR = ARTICLE_DIR / "reports"

for folder in [FIGURES_DIR, TABLES_DIR, REPORTS_DIR]:
    folder.mkdir(parents=True, exist_ok=True)

def sanitize_filename(name):
    return str(name).strip().replace(" ", "_").replace("/", "_")

def save_dataframe(df, filename, index=False):
    output_path = TABLES_DIR / sanitize_filename(filename)
    df.to_csv(output_path, index=index)
    return output_path

def save_excel(df, filename, index=False):
    output_path = TABLES_DIR / sanitize_filename(filename)
    df.to_excel(output_path, index=index)
    return output_path

def save_text_report(filename, text):
    output_path = REPORTS_DIR / sanitize_filename(filename)
    with open(output_path, "w", encoding="utf-8") as f:
        f.write(text)
    return output_path

def save_numpy_array(filename, arr):
    output_path = INTERMEDIATE_DIR / sanitize_filename(filename)
    np.save(output_path, arr)
    return output_path

SAVED_ARTIFACTS = []

def register_artifact(path_obj, description):
    SAVED_ARTIFACTS.append({
        "path": str(path_obj),
        "description": description
    })
    return path_obj


Experiment registry is ready.


## Text Cell 3 — Dataset Path Definition and Input Validation

This cell defines where the dataset should be read from and performs initial validation.  
The notebook is intentionally flexible: it can be adapted to CIC-IDS2017, UNSW-NB15, or another structured intrusion dataset, as long as the user provides a CSV file and identifies the target label column.

At this stage, the notebook checks:

- whether the dataset file exists,
- basic shape of the dataset,
- column names,
- missing values,
- label distribution,
- potential identifier columns useful later for forensic graph construction.

This early validation is important because downstream graph modeling and counterfactual analysis depend on stable and well-understood input data.

In [78]:
# ============================================
# Cell 3: Dataset configuration and loading
# ============================================

from pathlib import Path
import pandas as pd

# Use the actual dataset path
DATASET_PATH = Path("/content/drive/MyDrive/Datasets/Amal/ton-iot-train_test_network.csv")

# Preferred target column candidates (the code will auto-detect)
LABEL_CANDIDATES = ["type", "label", "attack", "class", "category"]

# Optional columns for forensic graph construction
SRC_IP_CANDIDATES = ["src_ip", "srcip", "src-ip", "src"]
DST_IP_CANDIDATES = ["dst_ip", "dstip", "dst-ip", "dst", "dest_ip"]
TIME_CANDIDATES = ["timestamp", "ts", "time", "datetime", "date"]

if not DATASET_PATH.exists():
    raise FileNotFoundError(
        f"Dataset not found at {DATASET_PATH}. "
        f"Please verify the Google Drive path."
    )

df = pd.read_csv(DATASET_PATH)

# Normalize column names for safer matching
original_columns = list(df.columns)
normalized_map = {col.strip().lower(): col for col in df.columns}

def find_first_matching_column(candidates, normalized_column_map):
    for candidate in candidates:
        key = candidate.strip().lower()
        if key in normalized_column_map:
            return normalized_column_map[key]
    return None

# Detect label column automatically
LABEL_COLUMN = find_first_matching_column(LABEL_CANDIDATES, normalized_map)
if LABEL_COLUMN is None:
    raise ValueError(
        "No valid label column was found in the dataset. "
        f"Tried: {LABEL_CANDIDATES}. Available columns: {original_columns}"
    )

# Detect optional forensic columns automatically
SRC_IP_COL = find_first_matching_column(SRC_IP_CANDIDATES, normalized_map)
DST_IP_COL = find_first_matching_column(DST_IP_CANDIDATES, normalized_map)
TIME_COL = find_first_matching_column(TIME_CANDIDATES, normalized_map)

dataset_summary = {
    "dataset_path": str(DATASET_PATH),
    "n_rows": int(df.shape[0]),
    "n_columns": int(df.shape[1]),
    "columns": original_columns,
    "missing_values_total": int(df.isna().sum().sum()),
    "label_column": LABEL_COLUMN,
}

label_distribution = df[LABEL_COLUMN].value_counts(dropna=False).to_dict()
dataset_summary["label_distribution"] = {
    str(k): int(v) for k, v in label_distribution.items()
}

identifier_availability = {
    "src_ip_present": SRC_IP_COL is not None,
    "dst_ip_present": DST_IP_COL is not None,
    "timestamp_present": TIME_COL is not None,
    "src_ip_column": SRC_IP_COL,
    "dst_ip_column": DST_IP_COL,
    "timestamp_column": TIME_COL,
}
dataset_summary["identifier_availability"] = identifier_availability

update_results("dataset", dataset_summary)
add_status_note("Dataset loaded successfully and basic validation completed.")
save_results_json()

print("Dataset loaded successfully.")
print("Dataset shape:", df.shape)
print("Detected label column:", LABEL_COLUMN)
print("Detected forensic columns:")
print("  SRC_IP_COL =", SRC_IP_COL)
print("  DST_IP_COL =", DST_IP_COL)
print("  TIME_COL   =", TIME_COL)
print("\nLabel distribution:")
print(pd.Series(dataset_summary["label_distribution"]))

Dataset loaded successfully.
Dataset shape: (211043, 44)
Detected label column: type
Detected forensic columns:
  SRC_IP_COL = src_ip
  DST_IP_COL = dst_ip
  TIME_COL   = None

Label distribution:
normal        50000
backdoor      20000
ddos          20000
dos           20000
injection     20000
password      20000
scanning      20000
ransomware    20000
xss           20000
mitm           1043
dtype: int64


## Text Cell 4 — Data Quality Inspection and Minimal Cleaning Strategy

This cell performs a practical cleaning stage suitable for a first strong baseline.  
The goal is not to over-engineer preprocessing, but to ensure that the model receives stable and reproducible input.

Typical operations include:

- removing duplicate rows,
- standardizing infinite values,
- handling missing values,
- separating numeric and categorical columns,
- preserving forensic identifiers (if present) before feature engineering.

The logic here is deliberately conservative:
a strong baseline should be clean, reproducible, and easy for a student to improve later.

In [79]:
# ============================================
# Cell 4: Data cleaning
# ============================================

df_clean = df.copy()

n_rows_before = len(df_clean)
df_clean = df_clean.drop_duplicates()
n_rows_after_dedup = len(df_clean)

df_clean = df_clean.replace([np.inf, -np.inf], np.nan)

# Preserve identifier columns separately
identifier_cols = [c for c in [SRC_IP_COL, DST_IP_COL, TIME_COL] if c in df_clean.columns]

# Basic missing-value treatment
for col in df_clean.columns:
    if col == LABEL_COLUMN:
        continue
    if df_clean[col].dtype == "object":
        df_clean[col] = df_clean[col].fillna("unknown")
    else:
        df_clean[col] = df_clean[col].fillna(df_clean[col].median())

cleaning_summary = {
    "n_rows_before": int(n_rows_before),
    "n_rows_after_deduplication": int(n_rows_after_dedup),
    "duplicates_removed": int(n_rows_before - n_rows_after_dedup),
    "identifier_columns": identifier_cols,
    "remaining_missing_values_total": int(df_clean.isna().sum().sum()),
}

update_results("preprocessing", cleaning_summary)
add_status_note("Data cleaning completed.")
save_results_json()

print("Cleaning summary:", cleaning_summary)

Cleaning summary: {'n_rows_before': 211043, 'n_rows_after_deduplication': 190474, 'duplicates_removed': 20569, 'identifier_columns': ['src_ip', 'dst_ip'], 'remaining_missing_values_total': 0}


## Text Cell 5 — Feature Preparation and Encoding

This cell prepares the feature matrix and target vector.  
It separates the machine learning input features from the label and from optional forensic identifier fields that should not be used as predictive shortcuts.

The preprocessing pipeline handles:

- numeric features,
- categorical features through one-hot encoding,
- robust assembly into a single feature matrix.

This separation is important because some columns may be highly informative for forensic reconstruction but inappropriate for direct predictive learning.

In [80]:
# ============================================
# Cell 5: Feature preparation (CORRECTED)
# ============================================

from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, LabelEncoder

# Strong protection against target leakage:
# remove the detected label column and any common alternative target columns
possible_target_like_cols = ["label", "type", "attack", "class", "category"]

feature_drop_cols = list(
    set([LABEL_COLUMN] + identifier_cols + [c for c in possible_target_like_cols if c in df_clean.columns])
)

X_raw = df_clean.drop(columns=feature_drop_cols, errors="ignore").copy()
y_raw = df_clean[LABEL_COLUMN].astype(str).copy()

label_encoder = LabelEncoder()
y = label_encoder.fit_transform(y_raw)

numeric_cols = X_raw.select_dtypes(include=[np.number]).columns.tolist()
categorical_cols = X_raw.select_dtypes(exclude=[np.number]).columns.tolist()

numeric_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
])

categorical_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore")),
])

preprocessor = ColumnTransformer(
    transformers=[
        ("num", numeric_transformer, numeric_cols),
        ("cat", categorical_transformer, categorical_cols),
    ]
)

feature_summary = {
    "dropped_columns": feature_drop_cols,
    "n_features_raw": int(X_raw.shape[1]),
    "n_numeric_features": len(numeric_cols),
    "n_categorical_features": len(categorical_cols),
    "numeric_columns": numeric_cols[:50],
    "categorical_columns": categorical_cols[:50],
    "target_classes": label_encoder.classes_.tolist(),
}

update_results("preprocessing", feature_summary)
add_status_note("Feature matrix and target vector prepared with leakage-safe column removal.")
save_results_json()

print("Dropped columns:", feature_drop_cols)
print("Raw feature count:", X_raw.shape[1])
print("Numeric columns:", len(numeric_cols))
print("Categorical columns:", len(categorical_cols))
print("Classes:", label_encoder.classes_)

Dropped columns: ['dst_ip', 'label', 'type', 'src_ip']
Raw feature count: 40
Numeric columns: 16
Categorical columns: 24
Classes: ['backdoor' 'ddos' 'dos' 'injection' 'mitm' 'normal' 'password'
 'ransomware' 'scanning' 'xss']


## Text Cell 6 — Stratified Train/Test Split and Forensic Context Preservation

This cell creates a stratified split so that each class remains proportionally represented in both training and test sets.  
It also preserves a parallel forensic context table containing source identifiers and timestamps when available.

This is especially important because later stages require us to map predictions back to:

- source and destination entities,
- temporal windows,
- suspicious interaction chains.

Therefore, the notebook splits both:
- the machine learning inputs, and
- the contextual forensic metadata.

In [81]:
# ============================================
# Cell 6: Train/test split
# ============================================

from sklearn.model_selection import train_test_split

context_df = df_clean[identifier_cols].copy() if identifier_cols else pd.DataFrame(index=df_clean.index)

X_train_raw, X_test_raw, y_train, y_test, ctx_train, ctx_test = train_test_split(
    X_raw,
    y,
    context_df,
    test_size=0.30,
    stratify=y,
    random_state=42
)

split_summary = {
    "train_size": int(len(X_train_raw)),
    "test_size": int(len(X_test_raw)),
    "test_ratio": 0.30,
    "stratified": True,
    "random_state": 42,
}

update_results("split", split_summary)
add_status_note("Train/test split completed with stratification.")
save_results_json()

print("Train size:", len(X_train_raw))
print("Test size:", len(X_test_raw))

Train size: 133331
Test size: 57143


## Text Cell 7 — Baseline Model Training

This cell trains a strong baseline classifier using a tree-based model.  
A tree ensemble is a practical choice here because it usually performs well on tabular intrusion datasets and supports downstream explainability with SHAP.

The model is embedded in a pipeline that combines:

- preprocessing,
- feature transformation,
- supervised classification.

This cell is designed to be strong enough for an initial publication-quality benchmark while remaining simple enough for the student to extend later.

In [82]:
# ============================================
# Cell 7: Model training (with diagnostics)
# ============================================

from sklearn.ensemble import RandomForestClassifier
from sklearn.pipeline import Pipeline
import numpy as np
import time

print("🔹 [START] Model training")

t0 = time.time()

# --------------------------------------------------
# Step 1: Initialize the Random Forest classifier
# --------------------------------------------------
print("🔸 Step 1: Initializing Random Forest...")

clf = RandomForestClassifier(
    n_estimators=200,
    max_depth=20,
    min_samples_split=10,
    min_samples_leaf=4,
    class_weight="balanced_subsample",
    n_jobs=-1,
    random_state=42
)

model = Pipeline(steps=[
    ("preprocessor", preprocessor),
    ("classifier", clf)
])

print("   Model initialized.")

# --------------------------------------------------
# Step 2: Train the pipeline
# --------------------------------------------------
print("🔸 Step 2: Training model...")
t_step = time.time()

model.fit(X_train_raw, y_train)

train_time = time.time() - t_step
print(f"   Training completed in {train_time:.2f} seconds")

# --------------------------------------------------
# Step 3: Extract trained components
# --------------------------------------------------
print("🔸 Step 3: Extracting trained components...")

trained_preprocessor = model.named_steps["preprocessor"]
trained_classifier = model.named_steps["classifier"]

# --------------------------------------------------
# Step 4: Recover transformed feature names
# --------------------------------------------------
print("🔸 Step 4: Recovering transformed feature names...")

try:
    feature_names = trained_preprocessor.get_feature_names_out()
    n_features = len(feature_names)
    print(f"   Number of transformed features: {n_features}")
except Exception as e:
    feature_names = None
    n_features = None
    print("   Could not recover transformed feature names.")
    print("   Reason:", repr(e))

# --------------------------------------------------
# Step 5: Tree-level diagnostics
# --------------------------------------------------
print("🔸 Step 5: Computing tree diagnostics...")

n_trees = len(trained_classifier.estimators_)
tree_depths = [estimator.tree_.max_depth for estimator in trained_classifier.estimators_]
tree_nodes = [estimator.tree_.node_count for estimator in trained_classifier.estimators_]

avg_depth = float(np.mean(tree_depths))
max_depth_observed = int(np.max(tree_depths))
min_depth_observed = int(np.min(tree_depths))

avg_nodes = float(np.mean(tree_nodes))
max_nodes = int(np.max(tree_nodes))
min_nodes = int(np.min(tree_nodes))

print(f"   Trees: {n_trees}")
print(f"   Depths -> avg: {avg_depth:.2f}, min: {min_depth_observed}, max: {max_depth_observed}")
print(f"   Nodes  -> avg: {avg_nodes:.2f}, min: {min_nodes}, max: {max_nodes}")

# --------------------------------------------------
# Step 6: Training accuracy diagnostic
# --------------------------------------------------
print("🔸 Step 6: Computing training accuracy...")
train_accuracy = model.score(X_train_raw, y_train)
print(f"   Training accuracy: {train_accuracy:.4f}")

# --------------------------------------------------
# Step 7: Native feature-importance diagnostics
# --------------------------------------------------
print("🔸 Step 7: Computing native feature importances...")

feature_importances = trained_classifier.feature_importances_
top_k = min(15, len(feature_importances))

if feature_names is not None:
    top_indices = np.argsort(feature_importances)[::-1][:top_k]
    top_features = [
        {
            "feature": str(feature_names[i]),
            "importance": float(feature_importances[i])
        }
        for i in top_indices
    ]
else:
    top_features = []

print("   Top feature importances computed.")

# --------------------------------------------------
# Step 8: Overfitting warning heuristic
# --------------------------------------------------
print("🔸 Step 8: Running simple overfitting diagnostics...")

diagnostic_notes = []

if train_accuracy >= 0.999:
    diagnostic_notes.append(
        "Training accuracy is extremely high; compare carefully with test metrics for overfitting assessment."
    )

if avg_depth >= 18:
    diagnostic_notes.append(
        "Average tree depth is relatively high; this may increase variance and reduce interpretability."
    )

if n_features is not None and n_features >= 800:
    diagnostic_notes.append(
        "The transformed feature space is large, which may increase computational cost for SHAP and LIME."
    )

if not diagnostic_notes:
    diagnostic_notes.append("No immediate diagnostic red flags were detected at training stage.")

print("   Diagnostic notes ready.")

# --------------------------------------------------
# Step 9: Save structured results
# --------------------------------------------------
print("🔸 Step 9: Saving model summary to results file...")

model_summary = {
    "model_name": "RandomForestClassifier",
    "hyperparameters": {
        "n_estimators": 200,
        "max_depth": 20,
        "min_samples_split": 10,
        "min_samples_leaf": 4,
        "class_weight": "balanced_subsample",
        "n_jobs": -1,
        "random_state": 42
    },
    "training_time_sec": float(train_time),
    "n_transformed_features": int(n_features) if n_features is not None else None,
    "tree_depth": {
        "avg_depth": avg_depth,
        "max_depth": max_depth_observed,
        "min_depth": min_depth_observed
    },
    "tree_nodes": {
        "avg_nodes": avg_nodes,
        "max_nodes": max_nodes,
        "min_nodes": min_nodes
    },
    "training_accuracy": float(train_accuracy),
    "top_feature_importances": top_features,
    "diagnostic_notes": diagnostic_notes
}

update_results("model", model_summary)
add_status_note("Baseline Random Forest model trained with diagnostics.")
save_results_json()

# --------------------------------------------------
# Step 10: Human-readable summary
# --------------------------------------------------
print("\n✅ Model training completed.")
print(f"⏱ Total time: {time.time() - t0:.2f} seconds")

print("\n📊 Model summary:")
print(f"   Model name            : RandomForestClassifier")
print(f"   Number of trees       : {n_trees}")
print(f"   Avg tree depth        : {avg_depth:.2f}")
print(f"   Max tree depth        : {max_depth_observed}")
print(f"   Min tree depth        : {min_depth_observed}")
print(f"   Avg node count        : {avg_nodes:.2f}")
print(f"   Max node count        : {max_nodes}")
print(f"   Min node count        : {min_nodes}")
print(f"   Transformed features  : {n_features}")
print(f"   Training accuracy     : {train_accuracy:.4f}")
print(f"   Training time (sec)   : {train_time:.2f}")

if top_features:
    print("\n🔝 Top important features:")
    for f in top_features[:10]:
        print(f"   - {f['feature']} ({f['importance']:.6f})")

print("\n📝 Diagnostic notes:")
for note in diagnostic_notes:
    print(f"   - {note}")

🔹 [START] Model training
🔸 Step 1: Initializing Random Forest...
   Model initialized.
🔸 Step 2: Training model...
   Training completed in 60.53 seconds
🔸 Step 3: Extracting trained components...
🔸 Step 4: Recovering transformed feature names...
   Number of transformed features: 880
🔸 Step 5: Computing tree diagnostics...
   Trees: 200
   Depths -> avg: 19.64, min: 8, max: 20
   Nodes  -> avg: 441.84, min: 65, max: 867
🔸 Step 6: Computing training accuracy...
   Training accuracy: 0.9724
🔸 Step 7: Computing native feature importances...
   Top feature importances computed.
🔸 Step 8: Running simple overfitting diagnostics...
   Diagnostic notes ready.
🔸 Step 9: Saving model summary to results file...

✅ Model training completed.
⏱ Total time: 64.01 seconds

📊 Model summary:
   Model name            : RandomForestClassifier
   Number of trees       : 200
   Avg tree depth        : 19.64
   Max tree depth        : 20
   Min tree depth        : 8
   Avg node count        : 441.84
   Max 

## Text Cell 8 — Core Predictive Evaluation

This cell computes the central predictive metrics needed to determine whether the current run is promising.  
Instead of producing many article figures immediately, the notebook writes the metrics and confusion matrix values directly into the intermediate result files.

The evaluation includes:

- accuracy,
- macro precision,
- macro recall,
- macro F1-score,
- confusion matrix,
- per-class performance summary.

These outputs are enough to decide whether the current model is already competitive or whether further improvement is required.

In [83]:
# ============================================
# Cell 8: Model evaluation
# ============================================

from sklearn.metrics import (
    accuracy_score,
    precision_recall_fscore_support,
    confusion_matrix,
    classification_report,
)

y_pred = model.predict(X_test_raw)
y_proba = model.predict_proba(X_test_raw)

acc = accuracy_score(y_test, y_pred)
precision_macro, recall_macro, f1_macro, _ = precision_recall_fscore_support(
    y_test, y_pred, average="macro", zero_division=0
)

cm = confusion_matrix(y_test, y_pred)
report_dict = classification_report(
    y_test,
    y_pred,
    target_names=label_encoder.classes_,
    output_dict=True,
    zero_division=0
)

evaluation_summary = {
    "accuracy": acc,
    "precision_macro": precision_macro,
    "recall_macro": recall_macro,
    "f1_macro": f1_macro,
    "confusion_matrix": cm.tolist(),
    "class_order": label_encoder.classes_.tolist(),
    "classification_report": report_dict,
}

update_results("evaluation", evaluation_summary)

quality_note = (
    f"Current macro-F1 = {f1_macro:.4f}. "
    + ("Promising baseline." if f1_macro >= 0.90 else "Needs improvement.")
)
add_status_note(quality_note)
save_results_json()

append_experiment_log({
    "run_id": RUN_ID,
    "timestamp": datetime.now().isoformat(),
    "model_name": "RandomForestClassifier",
    "accuracy": acc,
    "precision_macro": precision_macro,
    "recall_macro": recall_macro,
    "f1_macro": f1_macro,
    "n_train": len(X_train_raw),
    "n_test": len(X_test_raw),
})

print("Accuracy:", round(acc, 4))
print("Macro-F1:", round(f1_macro, 4))
print("Confusion matrix:")
print(cm)

Accuracy: 0.9706
Macro-F1: 0.9172
Confusion matrix:
[[ 5603     0     0     0     9     0     1     0     0     0]
 [    0  5604     2   144    43    68    19    24     0    94]
 [    0     5  5522     4   104     6     0     0    57     0]
 [    0    44     0  5677   187     1     0     0    26    54]
 [    0     3     0     9   296     1     0     0     2     1]
 [    0     6     0     5   120 12453     1     7     0    20]
 [    0     2     0     7    77     1  5848     0     0    24]
 [    0     0     0     0     0     1     0  4420     0     0]
 [    0     2     5     1    29     3     0     2  5958     0]
 [    0     5     0    93   353     8     0     1     0  4081]]


## Text Cell 9 — Probability Quality and Multi-Class ROC-AUC Assessment

Classification accuracy alone is not sufficient for a strong cybersecurity study.  
This cell evaluates how well the classifier separates classes probabilistically using one-vs-rest multi-class ROC-AUC.

This information is valuable because two models may have similar accuracy while differing substantially in confidence quality and class separability.
The resulting values are saved directly to the intermediate result file.

In [84]:
# ============================================
# Cell 9: ROC-AUC evaluation
# ============================================

from sklearn.metrics import roc_auc_score
from sklearn.preprocessing import label_binarize

classes_numeric = np.arange(len(label_encoder.classes_))
y_test_bin = label_binarize(y_test, classes=classes_numeric)

try:
    roc_auc_macro_ovr = roc_auc_score(y_test_bin, y_proba, average="macro", multi_class="ovr")
except Exception:
    roc_auc_macro_ovr = None

update_results("evaluation", {
    "roc_auc_macro_ovr": roc_auc_macro_ovr
})
save_results_json()

print("ROC-AUC macro OVR:", roc_auc_macro_ovr)

ROC-AUC macro OVR: 0.9967903601559851


## Text Cell 10 — Global Explainability with SHAP

This cell extracts global explanatory evidence using SHAP.  
The goal is to identify which transformed input features contribute most strongly to the model's decisions across the test set.

For the intermediate experimentation phase, the notebook stores:

- mean absolute SHAP importance,
- top-ranked influential features,
- a compact explanation summary.

This is enough to assess whether the model is learning meaningful attack-related patterns before generating final publication figures.

In [85]:
# ============================================
# Cell 10: Global explainability with safe fallback
# ============================================

import numpy as np
import pandas as pd
import time
from sklearn.inspection import permutation_importance

print("🔹 [START] Global explainability cell")
t0 = time.time()

preprocessor = model.named_steps["preprocessor"]
classifier = model.named_steps["classifier"]

# Transform test data
print("🔸 Transforming test data...")
X_test_transformed = preprocessor.transform(X_test_raw)
feature_names = preprocessor.get_feature_names_out()

if hasattr(X_test_transformed, "toarray"):
    X_test_transformed = X_test_transformed.toarray()

X_test_transformed = np.asarray(X_test_transformed, dtype=np.float32)

sample_n = min(1000, X_test_transformed.shape[0])
X_eval_sample = X_test_transformed[:sample_n]
y_eval_sample = np.asarray(y_test)[:sample_n]

explainability_summary = {
    "method_selected": None,
    "top_features": None,
}

# --------------------------------------------
# Attempt SHAP
# --------------------------------------------
use_shap = False
shap_importance_df = None

try:
    import shap

    print("🔸 Trying SHAP...")
    shap_sample_n = min(10, X_test_transformed.shape[0])
    X_shap_sample = X_test_transformed[:shap_sample_n]

    explainer = shap.TreeExplainer(classifier)
    shap_values_raw = explainer.shap_values(X_shap_sample, check_additivity=False)

    if isinstance(shap_values_raw, list):
        shap_array = np.stack(shap_values_raw, axis=0)
        mean_abs_shap = np.mean(np.abs(shap_array), axis=(0, 1))
    elif isinstance(shap_values_raw, np.ndarray) and shap_values_raw.ndim == 3:
        if shap_values_raw.shape[0] == X_shap_sample.shape[0]:
            mean_abs_shap = np.mean(np.abs(shap_values_raw), axis=(0, 2))
        else:
            mean_abs_shap = np.mean(np.abs(shap_values_raw), axis=(0, 1))
    else:
        mean_abs_shap = np.mean(np.abs(shap_values_raw), axis=0)

    shap_importance_df = pd.DataFrame({
        "feature": feature_names,
        "importance": mean_abs_shap
    }).sort_values("importance", ascending=False)

    max_importance = float(shap_importance_df["importance"].max())

    # sanity check
    if np.isfinite(max_importance) and max_importance < 1e6:
        use_shap = True
        explainability_summary["method_selected"] = "shap"
        print("   SHAP accepted.")
    else:
        print(f"   SHAP rejected due to unstable magnitude: {max_importance:.4e}")

except Exception as e:
    print("   SHAP failed:", repr(e))

# --------------------------------------------
# Fallback: permutation importance
# --------------------------------------------
if not use_shap:
    print("🔸 Using permutation importance fallback...")
    t_step = time.time()

    perm_result = permutation_importance(
        classifier,
        X_eval_sample,
        y_eval_sample,
        n_repeats=5,
        random_state=42,
        n_jobs=-1
    )

    shap_importance_df = pd.DataFrame({
        "feature": feature_names,
        "importance": perm_result.importances_mean
    }).sort_values("importance", ascending=False)

    explainability_summary["method_selected"] = "permutation_importance"
    print(f"   Done. Time: {time.time() - t_step:.2f} sec")

top_k = min(20, len(shap_importance_df))
top_features = shap_importance_df.head(top_k).to_dict(orient="records")

explainability_summary["top_features"] = [
    {"feature": str(row["feature"]), "importance": float(row["importance"])}
    for row in top_features
]

update_results("global_explainability", explainability_summary)
add_status_note(f"Global explainability completed using {explainability_summary['method_selected']}.")
save_results_json()

print("\n✅ Global explainability completed.")
print("⏱ Total time:", time.time() - t0)
print("\nTop features:")
print(shap_importance_df.head(10))

🔹 [START] Global explainability cell
🔸 Transforming test data...
🔸 Trying SHAP...
   SHAP accepted.

✅ Global explainability completed.
⏱ Total time: 2.4492626190185547

Top features:
                feature  importance
7     num__src_ip_bytes    0.016001
1         num__dst_port    0.015043
2         num__duration    0.012625
29  cat__conn_state_REJ    0.011292
3        num__src_bytes    0.010945
4        num__dst_bytes    0.010559
38   cat__conn_state_SF    0.010357
0         num__src_port    0.009027
9     num__dst_ip_bytes    0.008672
24    cat__service_http    0.008652


## Text Cell 11 — Local Explainability with LIME

This cell generates a local explanation for one representative test instance.  
Where SHAP provides global evidence, LIME helps interpret a single prediction in a human-readable way.

For the intermediate file strategy, the notebook stores:

- selected instance index,
- predicted class,
- true class,
- top local explanatory conditions.

This is particularly useful when analyzing difficult or suspicious cases for digital forensic interpretation.

In [86]:
# ============================================
# Cell 10.5: Dependency Management (LIME / SHAP / DiCE)
# ============================================

import sys
import subprocess

print("🔹 [START] Dependency check")

def ensure_package(pkg_name, import_name=None):
    if import_name is None:
        import_name = pkg_name.replace("-", "_")
    try:
        __import__(import_name)
        print(f"   ✅ {pkg_name} already installed")
    except ImportError:
        print(f"   ⚠️ Installing {pkg_name}...")
        subprocess.check_call([sys.executable, "-m", "pip", "install", pkg_name])
        print(f"   ✅ {pkg_name} installed")

# Core XAI dependencies
ensure_package("lime")
ensure_package("shap")
ensure_package("dice-ml", "dice_ml")

print("✅ All dependencies ready")

🔹 [START] Dependency check
   ✅ lime already installed
   ✅ shap already installed
   ✅ dice-ml already installed
✅ All dependencies ready


In [87]:
from lime.lime_tabular import LimeTabularExplainer

In [88]:
# ============================================
# Cell X: Install required XAI libraries
# ============================================

import sys
import subprocess

def install(package):
    subprocess.check_call([sys.executable, "-m", "pip", "install", package])

try:
    import lime
    print("LIME already installed.")
except ImportError:
    print("Installing LIME...")
    install("lime")

try:
    import shap
    print("SHAP already installed.")
except ImportError:
    print("Installing SHAP...")
    install("shap")

try:
    import dice_ml
    print("DiCE already installed.")
except ImportError:
    print("Installing DiCE...")
    install("dice-ml")

LIME already installed.
SHAP already installed.
DiCE already installed.


In [89]:
# ============================================
# Cell 11: LIME local explainability (FINAL ROBUST VERSION)
# ============================================

from lime.lime_tabular import LimeTabularExplainer
import numpy as np
import pandas as pd
import time

print("\n🔹 [START] LIME explainability")

t_start = time.time()

# --------------------------------------------
# Step 1: Prepare raw data
# --------------------------------------------
print("🔸 Step 1: Preparing raw data...")

X_train = X_train_raw.copy()
X_test = X_test_raw.copy()

feature_names = list(X_train.columns)
class_names = list(label_encoder.classes_)

print(f"   Train shape: {X_train.shape}")
print(f"   Test shape : {X_test.shape}")

# --------------------------------------------
# Step 2: Robust encoding of categorical features
# --------------------------------------------
print("🔸 Step 2: Encoding categorical features...")

categorical_cols = X_train.select_dtypes(include=["object", "category", "bool"]).columns.tolist()
numeric_cols = [c for c in feature_names if c not in categorical_cols]

categorical_indices = []
category_to_int = {}
int_to_category = {}
categorical_names = {}

for col in categorical_cols:
    train_vals = X_train[col].astype(str).fillna("MISSING")
    test_vals = X_test[col].astype(str).fillna("MISSING")

    train_categories = sorted(train_vals.unique().tolist())

    # Reserve -1 for unseen values
    mapping = {cat: idx for idx, cat in enumerate(train_categories)}
    reverse_mapping = {idx: cat for cat, idx in mapping.items()}

    X_train[col] = train_vals.map(mapping).astype(int)
    X_test[col] = test_vals.map(lambda v: mapping.get(v, -1)).astype(int)

    category_to_int[col] = mapping
    int_to_category[col] = reverse_mapping

    categorical_indices.append(feature_names.index(col))
    categorical_names[feature_names.index(col)] = train_categories + ["<UNSEEN>"]

# Ensure numeric columns are numeric and filled
numeric_fill_values = {}
for col in numeric_cols:
    X_train[col] = pd.to_numeric(X_train[col], errors="coerce")
    X_test[col] = pd.to_numeric(X_test[col], errors="coerce")

    fill_value = X_train[col].median()
    if pd.isna(fill_value):
        fill_value = 0.0

    numeric_fill_values[col] = float(fill_value)

    X_train[col] = X_train[col].fillna(fill_value)
    X_test[col] = X_test[col].fillna(fill_value)

X_train_np = X_train[feature_names].to_numpy(dtype=np.float64)
X_test_np = X_test[feature_names].to_numpy(dtype=np.float64)

print(f"   Categorical columns: {len(categorical_cols)}")
print(f"   Numeric columns    : {len(numeric_cols)}")

# --------------------------------------------
# Step 3: Prediction function with decoding
# --------------------------------------------
print("🔸 Step 3: Creating prediction function...")

def predict_fn(x):
    x_df = pd.DataFrame(x, columns=feature_names)

    for col in categorical_cols:
        def decode_val(v):
            try:
                iv = int(round(float(v)))
            except Exception:
                return "MISSING"
            if iv == -1:
                return "MISSING"
            return int_to_category[col].get(iv, "MISSING")

        x_df[col] = x_df[col].apply(decode_val)

    for col in numeric_cols:
        x_df[col] = pd.to_numeric(x_df[col], errors="coerce").fillna(numeric_fill_values[col])

    return model.predict_proba(x_df)

print("   Prediction function ready.")

# --------------------------------------------
# Step 4: Initialize LIME
# --------------------------------------------
print("🔸 Step 4: Initializing LIME explainer...")

lime_train_n = min(5000, len(X_train_np))
rng = np.random.default_rng(42)
lime_idx = rng.choice(len(X_train_np), size=lime_train_n, replace=False)
X_train_lime = X_train_np[lime_idx]

explainer = LimeTabularExplainer(
    training_data=X_train_lime,
    feature_names=feature_names,
    class_names=class_names,
    categorical_features=categorical_indices,
    categorical_names=categorical_names,
    mode="classification",
    discretize_continuous=True,
    random_state=42
)

print("   LIME explainer initialized.")

# --------------------------------------------
# Step 5: Explain one instance
# --------------------------------------------
print("🔸 Step 5: Generating explanation...")

idx = 0
instance = X_test_np[idx]

pred_probs = predict_fn(instance.reshape(1, -1))[0]
predicted_class_index = int(np.argmax(pred_probs))
predicted_class_name = class_names[predicted_class_index]
true_class_name = class_names[int(y_test[idx])]

exp = explainer.explain_instance(
    instance,
    predict_fn,
    labels=[predicted_class_index],
    num_features=10
)

lime_exp_list = exp.as_list(label=predicted_class_index)

print(f"   Predicted class: {predicted_class_name}")
print(f"   True class     : {true_class_name}")

print("\nTop LIME features:")
for f, w in lime_exp_list:
    print(f"   {f}: {w:.4f}")

# --------------------------------------------
# Step 6: Save results
# --------------------------------------------
update_results("xai", {
    "lime_local": {
        "instance_index": int(idx),
        "predicted_class_index": predicted_class_index,
        "predicted_class_name": predicted_class_name,
        "true_class_name": true_class_name,
        "top_features": [
            {"feature": str(f), "weight": float(w)}
            for f, w in lime_exp_list
        ]
    }
})

add_status_note("LIME explanation completed with robust unseen-category handling.")
save_results_json()

print("\n✅ LIME completed.")
print(f"⏱ Total time: {time.time() - t_start:.2f} sec")


🔹 [START] LIME explainability
🔸 Step 1: Preparing raw data...
   Train shape: (133331, 40)
   Test shape : (57143, 40)
🔸 Step 2: Encoding categorical features...
   Categorical columns: 24
   Numeric columns    : 16
🔸 Step 3: Creating prediction function...
   Prediction function ready.
🔸 Step 4: Initializing LIME explainer...
   LIME explainer initialized.
🔸 Step 5: Generating explanation...
   Predicted class: normal
   True class     : normal

Top LIME features:
   proto=udp: 0.1635
   service=dns: 0.1034
   weird_notice=-: -0.0732
   dns_RA=-: -0.0619
   ssl_resumed=-: -0.0571
   ssl_version=-: -0.0562
   http_resp_mime_types=-: -0.0537
   ssl_established=-: -0.0500
   dns_rejected=-: -0.0483
   ssl_cipher=-: -0.0414

✅ LIME completed.
⏱ Total time: 2.46 sec


## Text Cell 12 — Forensic Prediction Table Construction

This cell constructs a forensic working table by combining:

- contextual identifiers,
- ground truth labels,
- predicted labels,
- model confidence scores,
- hybrid suspicion indicators.

The suspicion modeling is intentionally risk-aware rather than uncertainty-only.  
The table therefore stores both:

- an entropy-based uncertainty signal, and
- a risk-versus-normal signal,

which are combined into a hybrid suspicion score. This intermediate table is central for downstream investigation because it links machine learning outcomes back to entities and contextual network evidence. It also provides the raw material needed for graph construction and suspicious-node ranking.


In [90]:
# ============================================
# Cell 12: Build forensic prediction table (FINAL CORRECTED)
# ============================================

from scipy.stats import entropy

# Maximum confidence of the predicted class
max_proba = y_proba.max(axis=1)

# Predicted and true labels in human-readable form
true_labels = [label_encoder.classes_[i] for i in y_test]
predicted_labels = [label_encoder.classes_[i] for i in y_pred]

forensic_table = ctx_test.copy() if not ctx_test.empty else pd.DataFrame(index=X_test_raw.index)
forensic_table = forensic_table.reset_index(drop=True)

forensic_table["true_label"] = true_labels
forensic_table["predicted_label"] = predicted_labels
forensic_table["predicted_confidence"] = max_proba

# Confidence-based uncertainty
entropy_scores = entropy(y_proba.T)

# Risk relative to the benign/normal class if available
if "normal" in label_encoder.classes_:
    normal_idx = list(label_encoder.classes_).index("normal")
    prob_normal = y_proba[:, normal_idx]
    risk_vs_normal = 1.0 - prob_normal
    forensic_table["normal_class_probability"] = prob_normal
else:
    prob_normal = np.zeros(len(y_proba))
    risk_vs_normal = 1.0 - max_proba
    forensic_table["normal_class_probability"] = np.nan

# Hybrid suspicion score: uncertainty + malicious likelihood
forensic_table["entropy_score"] = entropy_scores
forensic_table["risk_vs_normal"] = risk_vs_normal
forensic_table["suspicion_score"] = 0.5 * entropy_scores + 0.5 * risk_vs_normal

# Helpful flag for forensic review
forensic_table["is_misclassified"] = forensic_table["true_label"] != forensic_table["predicted_label"]

# Save a compact summary
forensic_summary = {
    "n_rows": int(len(forensic_table)),
    "mean_predicted_confidence": float(np.mean(forensic_table["predicted_confidence"])),
    "mean_entropy_score": float(np.mean(forensic_table["entropy_score"])),
    "mean_risk_vs_normal": float(np.mean(forensic_table["risk_vs_normal"])),
    "mean_suspicion_score": float(np.mean(forensic_table["suspicion_score"])),
    "misclassification_count": int(forensic_table["is_misclassified"].sum()),
}

update_results("forensics", {
    "prediction_table_summary": forensic_summary
})
add_status_note("Forensic prediction table constructed with hybrid suspicion scoring.")
save_results_json()

print("Forensic table created.")
print("Rows:", len(forensic_table))
print("Mean predicted confidence:", round(forensic_table["predicted_confidence"].mean(), 4))
print("Mean entropy score:", round(forensic_table["entropy_score"].mean(), 4))
print("Mean risk vs normal:", round(forensic_table["risk_vs_normal"].mean(), 4))
print("Mean suspicion score:", round(forensic_table["suspicion_score"].mean(), 4))
print("Misclassifications:", int(forensic_table["is_misclassified"].sum()))

forensic_table.head()


Forensic table created.
Rows: 57143
Mean predicted confidence: 0.6725
Mean entropy score: 1.2017
Mean risk vs normal: 0.811
Mean suspicion score: 1.0063
Misclassifications: 1681


,src_ip,dst_ip,true_label,predicted_label,predicted_confidence,normal_class_probability,entropy_score,risk_vs_normal,suspicion_score,is_misclassified
0,172.17.0.5,192.168.1.190,normal,normal,0.780474,0.780474,0.997679,0.219526,0.608602,False
1,192.168.1.193,192.168.1.33,backdoor,backdoor,0.667804,0.011850,1.303975,0.988150,1.146063,False
2,192.168.1.30,192.168.1.190,password,password,0.642239,0.015960,1.276055,0.984040,1.130048,False
3,192.168.1.33,192.168.1.193,ransomware,ransomware,0.737411,0.013880,1.108048,0.986120,1.047084,False
4,192.168.1.32,192.168.1.190,xss,xss,0.432268,0.016125,1.600655,0.983875,1.292265,False


## Text Cell 13 — Temporal Graph Construction for Digital Forensics

This cell transforms the forensic prediction table into a graph representation when source and destination identifiers are available.

The graph supports forensic questions such as:

- Which entities interact most frequently?
- Which nodes are linked to suspicious predictions?
- Which nodes appear central in malicious communication patterns?

A key design choice here is to accumulate both traffic frequency and suspicion, then normalize edge-level suspicion so that very frequent traffic does not automatically dominate the forensic graph. Even if timestamp information is not available, this graph still provides a strong and extensible baseline for digital forensic analysis.


In [91]:
# ============================================
# Cell 13: Temporal graph construction (FINAL CORRECTED)
# ============================================

import networkx as nx

G = nx.DiGraph()

graph_available = all(col in forensic_table.columns for col in [SRC_IP_COL, DST_IP_COL])

if graph_available:
    for _, row in forensic_table.iterrows():
        src = row[SRC_IP_COL]
        dst = row[DST_IP_COL]
        if pd.isna(src) or pd.isna(dst):
            continue

        suspicion_value = float(row["suspicion_score"])

        if G.has_edge(src, dst):
            G[src][dst]["count"] += 1
            G[src][dst]["suspicion_sum"] += suspicion_value
        else:
            G.add_edge(
                src,
                dst,
                count=1,
                suspicion_sum=suspicion_value
            )

    # Normalize edge suspicion so frequent traffic does not dominate unfairly
    for u, v, data in G.edges(data=True):
        data["suspicion_avg"] = data["suspicion_sum"] / max(data["count"], 1)

    n_nodes = G.number_of_nodes()
    n_edges = G.number_of_edges()
else:
    n_nodes, n_edges = 0, 0

update_results("forensics", {
    "graph_available": graph_available,
    "graph_n_nodes": int(n_nodes),
    "graph_n_edges": int(n_edges),
})
add_status_note(
    "Temporal interaction graph constructed with normalized edge suspicion."
    if graph_available else
    "Graph construction skipped because src/dst identifier columns are unavailable."
)
save_results_json()

print("Graph available:", graph_available)
print("Nodes:", n_nodes, "| Edges:", n_edges)


Graph available: True
Nodes: 518 | Edges: 674


## Text Cell 14 — Suspicious Node Ranking and Graph-Based Risk Analysis

This cell computes graph-driven risk indicators for entities appearing in the interaction graph.
A compact suspiciousness ranking is then written into the result file.

The ranking combines defensible graph and forensic signals such as:

- normalized outgoing suspiciousness,
- normalized incoming suspiciousness,
- degree-based prominence,
- PageRank centrality.

This produces a network-aware risk score rather than a simple traffic-volume score, which is more appropriate for forensic interpretation and later student extensions.


In [92]:
# ============================================
# Cell 14: Suspicious node ranking (FINAL CORRECTED)
# ============================================

node_ranking = []

if graph_available and G.number_of_nodes() > 0:
    pagerank_scores = nx.pagerank(G, alpha=0.85) if G.number_of_edges() > 0 else {n: 0.0 for n in G.nodes()}

    for node in G.nodes():
        out_suspicion = sum(G[node][nbr].get("suspicion_avg", 0.0) for nbr in G.successors(node))
        in_suspicion = sum(G[pred][node].get("suspicion_avg", 0.0) for pred in G.predecessors(node))
        out_degree = G.out_degree(node)
        in_degree = G.in_degree(node)
        pagerank = pagerank_scores.get(node, 0.0)

        # Normalized graph-aware risk score
        risk_score = (
            0.4 * out_suspicion +
            0.3 * in_suspicion +
            0.2 * pagerank +
            0.1 * (out_degree + in_degree)
        )

        node_ranking.append({
            "node": node,
            "out_suspicion": float(out_suspicion),
            "in_suspicion": float(in_suspicion),
            "out_degree": int(out_degree),
            "in_degree": int(in_degree),
            "pagerank": float(pagerank),
            "risk_score": float(risk_score),
        })

    node_ranking = sorted(node_ranking, key=lambda x: x["risk_score"], reverse=True)
    top_nodes = node_ranking[:20]
else:
    top_nodes = []

update_results("forensics", {
    "top_suspicious_nodes": top_nodes
})
add_status_note("Graph-based suspicious node ranking completed.")
save_results_json()

pd.DataFrame(top_nodes).head(10)


,node,out_suspicion,in_suspicion,out_degree,in_degree,pagerank,risk_score
0,192.168.1.190,85.676051,11.901754,156,14,0.007846,54.842516
1,192.168.1.31,88.739198,0.000000,69,0,0.001735,42.396026
2,192.168.1.30,74.324950,2.709406,60,2,0.002582,36.743318
3,192.168.1.34,65.772462,1.329913,53,1,0.002269,32.108413
4,192.168.1.152,47.978583,13.077920,55,11,0.003316,29.715472
5,192.168.1.195,42.044118,13.043900,39,11,0.003323,25.731481
6,2405:6e00:10ce:2c00:20c:29ff:feee:e07a,36.556870,0.000000,101,0,0.001735,24.723095
7,192.168.1.180,29.784022,1.121406,25,1,0.001796,14.850390
8,192.168.1.32,22.541066,0.000000,19,0,0.001735,10.916773
9,192.168.1.193,12.474493,10.184325,12,9,0.004760,10.146047


## Text Cell 15 — Counterfactual Cyber Defense

This cell produces counterfactual explanations for selected suspicious instances.
The purpose is to move beyond passive diagnosis and toward actionable cyber defense recommendations.

A counterfactual explanation answers the question:

**What minimal feature changes would alter the current prediction toward a safer target class?**

The implementation first checks whether `dice-ml` is available and installs it only if needed. It then selects a suspicious instance in a risk-aware way and generates a compact counterfactual record for the intermediate results file.


In [93]:
# ============================================
# Cell 15: Counterfactual analysis (FULL ROBUST VERSION)
# ============================================

import importlib.util
import subprocess
import sys
import pandas as pd
import numpy as np

counterfactual_summary = {"status": "not_run", "details": []}

try:
    # --------------------------------------------------
    # Step 1: Ensure DiCE is installed
    # --------------------------------------------------
    if importlib.util.find_spec("dice_ml") is None:
        print("Installing dice-ml...")
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "dice-ml"])

    import dice_ml
    from dice_ml import Dice

    print("🔹 [START] Counterfactual analysis")

    # --------------------------------------------------
    # Step 2: Prepare DiCE training data
    # --------------------------------------------------
    X_train_for_dice = X_train_raw.copy()
    X_test_for_dice = X_test_raw.copy()

    # DiCE requires the outcome column in the dataframe
    train_for_dice = X_train_for_dice.copy()
    train_for_dice[LABEL_COLUMN] = [label_encoder.classes_[i] for i in y_train]

    continuous_features = [
        c for c in X_train_for_dice.columns
        if pd.api.types.is_numeric_dtype(X_train_for_dice[c])
    ]

    print(f"   Training rows for DiCE: {len(train_for_dice)}")
    print(f"   Continuous features   : {len(continuous_features)}")

    # --------------------------------------------------
    # Step 3: Build DiCE data/model objects
    # --------------------------------------------------
    data_dice = dice_ml.Data(
        dataframe=train_for_dice,
        continuous_features=continuous_features,
        outcome_name=LABEL_COLUMN
    )

    model_dice = dice_ml.Model(model=model, backend="sklearn")
    exp_dice = Dice(data_dice, model_dice, method="genetic")

    # --------------------------------------------------
    # Step 4: Select a meaningful query instance
    # Prefer a suspicious non-normal instance
    # --------------------------------------------------
    if "risk_vs_normal" in forensic_table.columns and "predicted_label" in forensic_table.columns:
        candidate_mask = forensic_table["predicted_label"] != "normal"
        if candidate_mask.any():
            candidate_idx = forensic_table.loc[candidate_mask, "risk_vs_normal"].idxmax()
        else:
            candidate_idx = int(forensic_table["risk_vs_normal"].idxmax())
    else:
        candidate_idx = 0

    query_idx = int(candidate_idx)
    query_instance = X_test_for_dice.iloc[[query_idx]].copy()

    original_prediction = str(label_encoder.classes_[int(y_pred[query_idx])])
    all_classes = list(label_encoder.classes_)

    # --------------------------------------------------
    # Step 5: Define target class robustly
    # --------------------------------------------------
    if "normal" in all_classes:
        target_class_name = "normal"
    else:
        target_class_name = all_classes[0]

    print(f"   Query index          : {query_idx}")
    print(f"   Original prediction  : {original_prediction}")
    print(f"   Available classes    : {all_classes}")
    print(f"   Target class         : {target_class_name}")

    # If already normal, choose the most suspicious non-normal instance if possible
    if original_prediction == target_class_name and "predicted_label" in forensic_table.columns:
        non_normal_mask = forensic_table["predicted_label"] != target_class_name
        if non_normal_mask.any():
            query_idx = int(forensic_table.loc[non_normal_mask, "risk_vs_normal"].idxmax())
            query_instance = X_test_for_dice.iloc[[query_idx]].copy()
            original_prediction = str(label_encoder.classes_[int(y_pred[query_idx])])

            print("   Original query was already normal. Switched to a non-normal candidate.")
            print(f"   New query index      : {query_idx}")
            print(f"   New prediction       : {original_prediction}")

    # --------------------------------------------------
    # Step 6: Generate counterfactuals
    # --------------------------------------------------
    # Ensure object dtype for mixed-type tabular data
    query_instance = query_instance.astype(object)

    cf = exp_dice.generate_counterfactuals(
        query_instance,
        total_CFs=1,
        desired_class=target_class_name,
        features_to_vary="all"
    )

    cf_df = cf.cf_examples_list[0].final_cfs_df

    if cf_df is not None and not cf_df.empty:
        counterfactual_examples = cf_df.to_dict(orient="records")
        status_value = "success"
    else:
        counterfactual_examples = []
        status_value = "no_counterfactual_found"

    # --------------------------------------------------
    # Step 7: Save results
    # --------------------------------------------------
    counterfactual_summary = {
        "status": status_value,
        "query_index": int(X_test_for_dice.index[query_idx]),
        "original_prediction": original_prediction,
        "target_class": target_class_name,
        "available_classes": all_classes,
        "counterfactual_examples": counterfactual_examples
    }

except Exception as e:
    counterfactual_summary = {
        "status": "failed_or_skipped",
        "reason": str(e)
    }

update_results("counterfactuals", counterfactual_summary)
add_status_note(f"Counterfactual stage completed with status: {counterfactual_summary['status']}.")
save_results_json()

counterfactual_summary

🔹 [START] Counterfactual analysis
   Training rows for DiCE: 133331
   Continuous features   : 16
   Query index          : 31081
   Original prediction  : dos
   Available classes    : ['backdoor', 'ddos', 'dos', 'injection', 'mitm', 'normal', 'password', 'ransomware', 'scanning', 'xss']
   Target class         : normal


  0%|          | 0/1 [00:00<?, ?it/s]


{'status': 'failed_or_skipped',
 'reason': 'The target class for normal could not be identified'}

## Text Cell 16 — Automated Experiment Verdict

This cell creates a compact, human-readable decision summary based on the intermediate outputs.
Its purpose is to help the researcher quickly judge whether the run is already strong enough to justify deeper refinement.

The verdict is heuristic and can be adjusted, but it is useful for answering questions such as:

- Is the current macro-F1 already strong?
- Is ROC-AUC acceptable?
- Do we have meaningful global explainability evidence?
- Is the forensic graph active?
- Is counterfactual evidence available?

This step is especially useful when multiple notebook runs are compared over time.


In [94]:
# ============================================
# Cell 16: Automated verdict (CORRECTED)
# ============================================

f1_macro = experiment_results.get("evaluation", {}).get("f1_macro", None)
roc_auc_macro_ovr = experiment_results.get("evaluation", {}).get("roc_auc_macro_ovr", None)

global_xai = experiment_results.get("global_explainability", {})
top_global_features = global_xai.get("top_features", [])

graph_n_nodes = experiment_results.get("forensics", {}).get("graph_n_nodes", 0)
counterfactual_status = experiment_results.get("counterfactuals", {}).get("status", "not_run")

verdict = {
    "baseline_strength": "undetermined",
    "reasons": []
}

if f1_macro is not None:
    if f1_macro >= 0.95:
        verdict["reasons"].append("Excellent macro-F1.")
    elif f1_macro >= 0.90:
        verdict["reasons"].append("Strong macro-F1.")
    else:
        verdict["reasons"].append("Macro-F1 should be improved.")

if roc_auc_macro_ovr is not None:
    if roc_auc_macro_ovr >= 0.95:
        verdict["reasons"].append("Excellent probabilistic class separability.")
    elif roc_auc_macro_ovr >= 0.90:
        verdict["reasons"].append("Promising ROC-AUC.")
    else:
        verdict["reasons"].append("ROC-AUC should be improved.")

if len(top_global_features) >= 10:
    verdict["reasons"].append("Global explainability evidence is available.")
else:
    verdict["reasons"].append("Global explainability evidence is limited.")

if graph_n_nodes and graph_n_nodes > 0:
    verdict["reasons"].append("Forensic graph layer is active.")
else:
    verdict["reasons"].append("Forensic graph layer is not yet active.")

if counterfactual_status == "success":
    verdict["reasons"].append("Counterfactual defense evidence is available.")
elif counterfactual_status == "failed_or_skipped":
    verdict["reasons"].append("Counterfactual stage still needs completion.")

positive_count = sum(
    any(keyword in r for keyword in ["Excellent", "Strong", "Promising", "available", "active"])
    for r in verdict["reasons"]
)

verdict["baseline_strength"] = (
    "strong" if positive_count >= 4 else
    "moderate" if positive_count >= 2 else
    "weak"
)

update_results("evaluation", {"automated_verdict": verdict})
save_results_json()

print(json.dumps(verdict, indent=2))

{
  "baseline_strength": "strong",
  "reasons": [
    "Strong macro-F1.",
    "Excellent probabilistic class separability.",
    "Global explainability evidence is available.",
    "Forensic graph layer is active.",
    "Counterfactual stage still needs completion."
  ]
}


## Text Cell 17 — Final Persistence of Intermediate Outputs

This last cell ensures that all accumulated results are safely persisted to Google Drive.  
It also prints the locations of the two main files so that the researcher or student can inspect them immediately after execution.

At this stage, the notebook has **not** generated a heavy set of final article plots.
Instead, it has produced a compact experimental memory of the run that can later be converted into publication-quality assets.

## Text Cell 17 — Article-Ready Figures and Tables

This section transforms the intermediate experimental outputs into **publication-ready assets** saved directly to Google Drive.  
It produces key figures and tables that are useful for the manuscript, including:

- a confusion-matrix figure,
- a per-class metrics table,
- a global feature-importance figure,
- a suspicious-node ranking table,
- a suspicious-flow table, and
- a compact text report summarizing the main findings.

All generated files are saved into the structured Google Drive folders created at the beginning of the notebook.


In [ ]:

# ============================================
# Cell 17A: Generate and save article figures/tables
# ============================================

import matplotlib.pyplot as plt
import pandas as pd
import numpy as np

generated_paths = []

# 1) Per-class classification table
report_df = pd.DataFrame(report_dict).transpose().reset_index().rename(columns={"index": "class_or_metric"})
report_csv = register_artifact(save_dataframe(report_df, "classification_report.csv", index=False), "Per-class classification report (CSV)")
generated_paths.append(report_csv)

try:
    report_xlsx = register_artifact(save_excel(report_df, "classification_report.xlsx", index=False), "Per-class classification report (Excel)")
    generated_paths.append(report_xlsx)
except Exception as e:
    print("Excel export skipped:", e)

# 2) Confusion matrix table
cm_df = pd.DataFrame(cm, index=label_encoder.classes_, columns=label_encoder.classes_)
cm_csv = register_artifact(save_dataframe(cm_df.reset_index().rename(columns={"index": "true_pred"}), "confusion_matrix_table.csv", index=False), "Confusion matrix table (CSV)")
generated_paths.append(cm_csv)

# 3) Confusion matrix figure
plt.figure(figsize=(8, 6))
plt.imshow(cm, interpolation="nearest")
plt.title("Confusion Matrix")
plt.colorbar()
tick_marks = np.arange(len(label_encoder.classes_))
plt.xticks(tick_marks, label_encoder.classes_, rotation=45, ha="right")
plt.yticks(tick_marks, label_encoder.classes_)
plt.xlabel("Predicted Label")
plt.ylabel("True Label")

thresh = cm.max() / 2.0 if cm.size else 0
for i in range(cm.shape[0]):
    for j in range(cm.shape[1]):
        plt.text(j, i, format(cm[i, j], "d"),
                 ha="center", va="center",
                 color="white" if cm[i, j] > thresh else "black")

plt.tight_layout()
cm_fig_path = FIGURES_DIR / "confusion_matrix.png"
plt.savefig(cm_fig_path, dpi=300, bbox_inches="tight")
plt.close()
register_artifact(cm_fig_path, "Confusion matrix figure")
generated_paths.append(cm_fig_path)

# 4) Global explainability figure
global_xai = experiment_results.get("global_explainability", experiment_results.get("xai", {}))
top_features = global_xai.get("top_features", [])

if top_features:
    xai_df = pd.DataFrame(top_features)
    xai_csv = register_artifact(save_dataframe(xai_df, "global_feature_importance.csv", index=False), "Global feature importance table")
    generated_paths.append(xai_csv)

    plot_df = xai_df.head(15).copy()
    value_col = "importance" if "importance" in plot_df.columns else plot_df.columns[-1]
    feature_col = "feature" if "feature" in plot_df.columns else plot_df.columns[0]

    plt.figure(figsize=(9, 6))
    plt.barh(plot_df[feature_col].astype(str)[::-1], plot_df[value_col][::-1])
    plt.title("Top Global Features")
    plt.xlabel("Importance")
    plt.ylabel("Feature")
    plt.tight_layout()
    xai_fig_path = FIGURES_DIR / "global_feature_importance.png"
    plt.savefig(xai_fig_path, dpi=300, bbox_inches="tight")
    plt.close()
    register_artifact(xai_fig_path, "Global feature importance figure")
    generated_paths.append(xai_fig_path)

# 5) Suspicious node ranking table
top_nodes = experiment_results.get("forensics", {}).get("top_suspicious_nodes", [])
if top_nodes:
    nodes_df = pd.DataFrame(top_nodes)
    nodes_csv = register_artifact(save_dataframe(nodes_df, "top_suspicious_nodes.csv", index=False), "Top suspicious nodes table")
    generated_paths.append(nodes_csv)

# 6) Suspicious flow table
if "forensic_table" in globals() and isinstance(forensic_table, pd.DataFrame) and not forensic_table.empty:
    suspicious_flows_df = forensic_table.sort_values("suspicion_score", ascending=False).head(100).copy()
    suspicious_csv = register_artifact(save_dataframe(suspicious_flows_df, "top_suspicious_flows.csv", index=False), "Top suspicious flows table")
    generated_paths.append(suspicious_csv)

# 7) LIME table if available
if "lime_df" in globals() and isinstance(lime_df, pd.DataFrame) and not lime_df.empty:
    lime_csv = register_artifact(save_dataframe(lime_df, "lime_local_explanation.csv", index=False), "LIME local explanation table")
    generated_paths.append(lime_csv)

# 8) Counterfactual table if available
if "counterfactual_df" in globals() and isinstance(counterfactual_df, pd.DataFrame) and not counterfactual_df.empty:
    cf_csv = register_artifact(save_dataframe(counterfactual_df, "counterfactual_examples.csv", index=False), "Counterfactual examples table")
    generated_paths.append(cf_csv)

print("Generated article assets:")
for p in generated_paths:
    print(" -", p)


## Text Cell 18 — Compact Research Report and Drive Summary

This final production cell writes a concise text report summarizing the main results and prints the exact Google Drive locations of all exported assets.  
This makes it easy to retrieve the figures and tables later when writing the manuscript.


In [ ]:

# ============================================
# Cell 17B: Save compact report and print artifact registry
# ============================================

summary_lines = [
    "Forensic-Ready Explainable IDS — Compact Result Report",
    "=" * 58,
    f"Run ID: {experiment_results.get('run_id', 'N/A')}",
    f"Created at: {experiment_results.get('created_at', 'N/A')}",
    "",
    "Core predictive metrics:",
    f"- Accuracy: {experiment_results.get('evaluation', {}).get('accuracy', 'N/A')}",
    f"- Macro precision: {experiment_results.get('evaluation', {}).get('precision_macro', 'N/A')}",
    f"- Macro recall: {experiment_results.get('evaluation', {}).get('recall_macro', 'N/A')}",
    f"- Macro F1: {experiment_results.get('evaluation', {}).get('f1_macro', 'N/A')}",
    f"- ROC-AUC macro OVR: {experiment_results.get('evaluation', {}).get('roc_auc_macro_ovr', 'N/A')}",
    "",
    "Forensic graph summary:",
    f"- Graph available: {experiment_results.get('forensics', {}).get('graph_available', 'N/A')}",
    f"- Number of nodes: {experiment_results.get('forensics', {}).get('graph_n_nodes', 'N/A')}",
    f"- Number of edges: {experiment_results.get('forensics', {}).get('graph_n_edges', 'N/A')}",
    "",
    "Counterfactual status:",
    f"- {experiment_results.get('counterfactuals', {}).get('status', 'N/A')}",
    "",
    "Saved artifacts:",
]

for item in SAVED_ARTIFACTS:
    summary_lines.append(f"- {item['description']}: {item['path']}")

summary_text = "\n".join(summary_lines)
summary_report_path = register_artifact(save_text_report("summary_report.txt", summary_text), "Compact summary report")

update_results("artifact_registry", SAVED_ARTIFACTS)
save_results_json()

print(summary_text)
print("\nSummary report saved to:")
print(summary_report_path)


In [95]:
# ============================================
# Cell 17: Final save and file locations
# ============================================

save_results_json()

print("Main intermediate files generated:")
print(f"1. Structured results JSON: {RESULTS_JSON}")
print(f"2. Flat experiment log CSV: {RESULTS_CSV}")

print("\nSuggested next step:")
print("Review the JSON and CSV outputs first. If the results are strong enough, create a second notebook dedicated to figures and tables for the article.")

print("\nSaved article/report artifacts:")
for item in SAVED_ARTIFACTS:
    print(f"- {item['description']}: {item['path']}")


Main intermediate files generated:
1. Structured results JSON: /content/drive/MyDrive/Outputs/Forensic_Ready_XAI_IDS/intermediate_outputs/experiment_results.json
2. Flat experiment log CSV: /content/drive/MyDrive/Outputs/Forensic_Ready_XAI_IDS/intermediate_outputs/experiment_log.csv

Suggested next step:
Review the JSON and CSV outputs first. If the results are strong enough, create a second notebook dedicated to figures and tables for the article.
